# Part 3: Add a non-authenticated MCP source (Microsoft Learn)

In the previous parts, you built a knowledge base with indexed data and saw how it's exposed as an MCP endpoint. Now you'll go in the other direction — connecting an **external MCP server** as a live knowledge source in your knowledge base.

The [Microsoft Learn MCP Server](https://learn.microsoft.com/en-us/training/support/mcp) (`https://learn.microsoft.com/api/mcp`) provides free, unauthenticated access to Microsoft's official documentation. It exposes tools like `microsoft_docs_search`, `microsoft_docs_fetch`, and `microsoft_code_sample_search`.

When you add this as an MCP knowledge source, your knowledge base can send queries to both your indexed enterprise content **and** live Microsoft documentation — enabling agentic retrieval across internal and external sources.

> ⚠️ **Note**: The Azure AI Search Python SDK (`azure-search-documents==11.7.0b2`) does not yet have classes for MCP knowledge sources. This notebook uses the REST API via the `requests` library.

## Step 1: Load environment variables

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
api_key = os.environ["AZURE_SEARCH_ADMIN_KEY"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_openai_key = os.environ["AZURE_OPENAI_KEY"]
azure_openai_chatgpt_deployment = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
azure_openai_chatgpt_model_name = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

knowledge_base_name = "hr-and-health-docs-knowledge-base"
api_version = "2025-11-01-preview"

print("Environment variables loaded")

## Step 2: Create Learn MCP knowledge source

Create a knowledge source that points to the Microsoft Learn MCP server. This is a `mcpTool` kind that defines:
- `serverURL`: The MCP server endpoint
- `toolName`: Which MCP tool to use for retrieval

No authentication headers are needed — the Learn MCP server is publicly available.

In [ ]:
import requests

learn_ks_name = "learn-mcp-source"

learn_ks_body = {
    "name": learn_ks_name,
    "kind": "mcpTool",
    "description": "Microsoft Learn documentation — search official Azure and Microsoft docs in real time",
    "mcpToolParameters": {
        "serverURL": "https://learn.microsoft.com/api/mcp",
        "toolName": "microsoft_docs_search"
    }
}

url = f"{endpoint}/knowledgesources/{learn_ks_name}?api-version={api_version}"
headers = {"Content-Type": "application/json", "api-key": api_key}

response = requests.put(url, json=learn_ks_body, headers=headers)
response.raise_for_status()
print(f"Knowledge source '{learn_ks_name}' created: {response.status_code}")

## Step 3: Add Learn source to the knowledge base

Update the existing knowledge base to include the Learn MCP source alongside the indexed hrdocs and healthdocs. The knowledge base can now choose to send the queries to any combination of these sources based on the retrieval instructions.

Notice the updated `retrievalInstructions` — they guide the agent on when to use the Learn source versus the indexed enterprise content.

In [ ]:
kb_body = {
    "name": knowledge_base_name,
    "description": "Knowledge base combining Zava enterprise docs with live Microsoft Learn documentation",
    "retrievalInstructions": (
        "Use healthdocs for health benefits and insurance questions. "
        "Use hrdocs for HR policies and company information. "
        "Use learn-mcp-source for Azure, Microsoft product, or technical documentation questions."
    ),
    "outputMode": "answerSynthesis",
    "knowledgeSources": [
        {"name": "healthdocs-knowledge-source"},
        {"name": "hrdocs-knowledge-source"},
        {"name": learn_ks_name}
    ],
    "models": [
        {
            "kind": "azureOpenAI",
            "azureOpenAIParameters": {
                "resourceUri": azure_openai_endpoint,
                "deploymentId": azure_openai_chatgpt_deployment,
                "modelName": azure_openai_chatgpt_model_name,
                "apiKey": azure_openai_key
            }
        }
    ]
}

url = f"{endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}"
response = requests.put(url, json=kb_body, headers=headers)
response.raise_for_status()
print(f"Knowledge base '{knowledge_base_name}' updated: {response.status_code}")

## Step 4: Query across indexed + live sources

Now ask a question that requires the Learn MCP source. The knowledge base will send the query to the Learn server, since it asks specifically about Azure documentation.

In [ ]:
from IPython.display import Markdown, display

retrieve_url = f"{endpoint}/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}"

retrieve_body = {
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": "What is agentic retrieval in Azure AI Search?"}]
        }
    ],
    "includeActivity": True
}

response = requests.post(retrieve_url, json=retrieve_body, headers=headers)
try:
    response.raise_for_status()
    result = response.json()
    display(Markdown(result["response"][0]["content"][0]["text"]))
except requests.RequestException as e:
    print(f"Error occurred: {e}")
    print(response.json())

## Step 5: Verify source selection

Check the activity log to confirm the knowledge base selected the Learn MCP source for this Azure-related question:

In [ ]:
import json

for activity in result.get("activity", []):
    ks_name = activity.get("knowledgeSourceName", "")
    activity_type = activity.get("type", "")
    if ks_name:
        print(f"  [{activity_type}] → {ks_name}")

print("\nFull activity log:")
print(json.dumps(result.get("activity", []), indent=2))

## Step 6: Mixed query: Enterprise + Learn

Ask a question that requires both enterprise data and Microsoft documentation. The knowledge base should select multiple sources to construct a complete answer:

In [ ]:
retrieve_body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "What are Zava's health insurance options? "
                        "Also, how do I create a knowledge base in Azure AI Search?"
                    )
                }
            ]
        }
    ],
    "includeActivity": True
}

response = requests.post(retrieve_url, json=retrieve_body, headers=headers)
result = None
try:
    response.raise_for_status()
    result = response.json()
    display(Markdown(result["response"][0]["content"][0]["text"]))
except requests.RequestException as e:
    print(f"Error occurred: {e}")
    print(response.json())

print("\nSources queried:")
for activity in result.get("activity", []):
    ks_name = activity.get("knowledgeSourceName", "")
    if ks_name:
        print(f"  → {ks_name}")

## Summary

You've added a live MCP knowledge source to your knowledge base! The Microsoft Learn MCP server provides real-time access to Azure documentation without any indexing or authentication setup.

**Key concepts:**
- MCP knowledge sources use `kind: mcpTool` with a `serverURL` and `toolName`
- The Learn MCP server (`https://learn.microsoft.com/api/mcp`) requires no authentication
- The knowledge base selects appropriate sources (indexed, MCP, or both) using retrieval instructions
- Activity logs show which sources were queried for each request
- The SDK doesn't support MCP sources yet — use the REST API with `requests`

➡️ Continue to [Part 4: Add an authenticated MCP source (GitHub)](part4-mcp-authenticated.ipynb) to connect a GitHub MCP server that requires Bearer token authentication.